# 数据预处理与可视化

数据预处理是机器学习中最关键的一步。垃圾进，垃圾出！

本 Notebook 涵盖：数据探索、缺失值处理、特征缩放、降维可视化。

## 1. 环境准备与数据加载

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils.helpers import describe_data, load_dataset

%matplotlib inline

In [ ]:
df, desc = load_dataset('wine')
print(f"数据集: Wine 葡萄酒分类")
print(f"{desc[:200]}...")

## 2. 数据探索 (EDA)

In [ ]:
describe_data(df)

In [ ]:
df['target'].value_counts().plot(kind='bar')
plt.title('Target Class Distribution')
plt.show()

## 3. 缺失值处理策略

In [ ]:
np.random.seed(42)
df_demo = pd.DataFrame({
    'age': np.random.normal(35, 10, 1000),
    'income': np.random.lognormal(10, 0.5, 1000),
    'city': np.random.choice(['北京', '上海', '广州'], 1000)
})
mask = np.random.random(df_demo.shape) < 0.1
df_demo = df_demo.mask(mask)
print(f"缺失值统计:\n{df_demo.isnull().sum()}")
print(f"\n缺失比例: {df_demo.isnull().sum() / len(df_demo) * 100:.1f}%")

In [ ]:
def handle_missing(df, strategy='mean'):
    df_clean = df.copy()
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    if strategy == 'mean':
        df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].mean())
    elif strategy == 'median':
        df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median())
    elif strategy == 'mode':
        df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].mode().iloc[0])
        for col in categorical_cols:
            df_clean[col] = df_clean[col].fillna(df_clean[col].mode().iloc[0])
    return df_clean

print(f"均值填充后 - 年龄均值: {handle_missing(df_demo, 'mean')['age'].mean():.2f}")
print(f"中位数填充后 - 年龄均值: {handle_missing(df_demo, 'median')['age'].mean():.2f}")

## 4. 特征缩放

为什么要缩放？因为很多算法（SVM、KNN、神经网络）对特征的量纲敏感。

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

data = np.array([[1], [2], [3], [4], [5], [6], [7], [8], [9], [10]])

standard_scaler = StandardScaler()
standardized = standard_scaler.fit_transform(data)
print(f"标准化 (Z-score): {standardized.T}")
print(f"  均值={standardized.mean():.4f}, 标准差={standardized.std():.4f}")

minmax_scaler = MinMaxScaler()
normalized = minmax_scaler.fit_transform(data)
print(f"\n归一化 (MinMax): {normalized.T}")
print(f"  范围: [{normalized.min():.2f}, {normalized.max():.2f}]")

## 5. 数据可视化

### 5.1 特征分布

In [ ]:
from utils.helpers import plot_distributions

feature_cols = [c for c in df.columns if c != 'target']
plot_distributions(df, cols=feature_cols[:6])

### 5.2 相关矩阵

In [ ]:
from utils.helpers import plot_correlation_matrix

plot_correlation_matrix(df[feature_cols])

### 5.3 PCA 降维可视化

将高维数据降到 2 维，直观观察数据的分布。

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = df[feature_cols]
y = df['target']

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"各主成分方差解释比: {pca.explained_variance_ratio_}")
print(f"累计方差解释比: {pca.explained_variance_ratio_.sum():.2%}")

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['target'].astype('category').cat.codes,
                      cmap='viridis', alpha=0.7, edgecolors='k')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
plt.title('PCA - Wine Dataset')
plt.colorbar(scatter, label='Wine Class')
plt.show()

## 练习

1. 用 `load_dataset('breast_cancer')` 加载乳腺癌数据集，做完整的 EDA
2. 对比 StandardScaler 和 MinMaxScaler 对同一数据的影响
3. 尝试对乳腺癌数据做 PCA 可视化，观察良恶性是否可分